# Chapter 15 — Problem Set 1: Solutions

Reference solutions for the exercises in `problem_set_1.ipynb`. These run **offline** with only the dependencies in `requirements.txt`.

---
*Generated by Berta AI*

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'scripts'))

import time
import json
from pathlib import Path

import numpy as np
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from pydantic import BaseModel, Field, ConfigDict, ValidationError
from fastapi import FastAPI
from fastapi.testclient import TestClient

print('imports OK')

## 1. Package a Model with joblib

In [ ]:
rng = np.random.default_rng(0)
X = rng.normal(size=(200, 2))
y = (X[:, 0] + 0.5 * X[:, 1] > 0).astype(int)

pipe = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=200))])
pipe.fit(X, y)

models_dir = Path('../../models'); models_dir.mkdir(exist_ok=True)
artifact = models_dir / 'my_model.joblib'
joblib.dump(pipe, artifact)

reloaded = joblib.load(artifact)
assert (reloaded.predict(X) == pipe.predict(X)).all()
print('round-trip OK; artifact size:', artifact.stat().st_size, 'bytes')

## 2. Pydantic Schemas

In [ ]:
class PredictRequest(BaseModel):
    model_config = ConfigDict(extra='forbid')
    feature_a: float = Field(..., description='Numeric feature A.')
    feature_b: float = Field(..., description='Numeric feature B.')

class PredictResponse(BaseModel):
    prediction: float
    model_version: str

ok = PredictRequest(feature_a=0.5, feature_b=-0.3)
print('valid:', ok.model_dump())

for bad in [{'feature_a': 0.1, 'feature_b': 0.0, 'extra': 1},
            {'feature_a': 'x',  'feature_b': 0.0}]:
    try:
        PredictRequest(**bad)
    except ValidationError as e:
        print('rejected:', e.errors()[0]['type'])

## 3. /predict Endpoint

In [ ]:
app = FastAPI()
MODEL = reloaded
VERSION = '0.1.0'

@app.post('/predict', response_model=PredictResponse)
def predict(req: PredictRequest) -> PredictResponse:
    x = np.asarray([[req.feature_a, req.feature_b]], dtype=float)
    y = MODEL.predict(x)[0]
    return PredictResponse(prediction=float(y), model_version=VERSION)

client = TestClient(app)
r = client.post('/predict', json={'feature_a': 0.4, 'feature_b': -0.1})
print('status:', r.status_code, 'body:', r.json())
assert r.status_code == 200
assert set(r.json().keys()) == {'prediction', 'model_version'}

bad = client.post('/predict', json={'feature_a': 'oops', 'feature_b': 0.0})
print('bad status:', bad.status_code)
assert bad.status_code == 422

## 4. Dockerfile

In [ ]:
DOCKERFILE = '''
# syntax=docker/dockerfile:1.6
FROM python:3.11-slim AS builder
WORKDIR /build
COPY requirements.txt .
RUN pip install --user --no-cache-dir -r requirements.txt

FROM python:3.11-slim
WORKDIR /app
COPY --from=builder /root/.local /root/.local
ENV PATH=/root/.local/bin:$PATH
COPY scripts ./scripts
COPY models ./models

RUN useradd --uid 10001 --no-create-home appuser
USER appuser

EXPOSE 8000
HEALTHCHECK --interval=15s --timeout=3s --retries=3 \\
    CMD python -c "import httpx, sys; sys.exit(0 if httpx.get('http://localhost:8000/health').json().get('ready') else 1)"

CMD ["uvicorn", "scripts.deployment:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]
'''.strip()
print(DOCKERFILE)

## 5. Batch Predictions

In [ ]:
class BatchRequest(BaseModel):
    model_config = ConfigDict(extra='forbid')
    records: list[PredictRequest]

@app.post('/predict/batch')
def predict_batch(req: BatchRequest):
    x = np.asarray([[r.feature_a, r.feature_b] for r in req.records], dtype=float)
    y = MODEL.predict(x)
    return {'predictions': [float(v) for v in y], 'n': len(req.records)}

records = [{'feature_a': float(rng.normal()), 'feature_b': float(rng.normal())} for _ in range(100)]

t0 = time.perf_counter()
for r in records:
    client.post('/predict', json=r)
t_single = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
client.post('/predict/batch', json={'records': records})
t_batch = (time.perf_counter() - t0) * 1000

print(f'single: {t_single:.1f}ms total, batch: {t_batch:.1f}ms total, speedup: {t_single/max(t_batch,1e-6):.1f}x')

## 6. /version and /health

In [ ]:
class VersionResponse(BaseModel):
    name: str
    version: str
    stage: str
    framework: str

class HealthResponse(BaseModel):
    status: str
    ready: bool

@app.get('/version', response_model=VersionResponse)
def version():
    return VersionResponse(name='demo', version=VERSION, stage='Staging', framework='sklearn')

@app.get('/health', response_model=HealthResponse)
def health():
    return HealthResponse(status='ok', ready=True)

v = client.get('/version').json()
h = client.get('/health').json()
print('version:', v); print('health:', h)
assert isinstance(v['name'], str) and isinstance(v['version'], str)
assert isinstance(h['ready'], bool)
print('\nLiveness vs readiness: liveness = process answers; readiness = model loaded.')